# nadakit

> Wav2vec 2.0 on Modular MAX, plus the embed engine and FastHTML app from nadayantra.


#| export
[Wav2vec 2.0](https://arxiv.org/abs/2006.11477) on [Modular MAX](https://docs.modular.com/max/). This package combines the `nada` model API with the embed runtime that `nadayantra` and the gananayantra helper each used to keep a copy of.

An embedding is the mean final hidden state over the clip, minus a fixed typical-audio vector (`embed_center.npy`), scaled to unit length.


## Install

```sh
pixi install
pixi run convert
```

`convert` downloads `facebook/wav2vec2-base-960h` and writes `~/.cache/nadakit/models/wav2vec2.safetensors` (or `$NADA_MODELS`). The Hugging Face file is already safetensors; conversion folds the position convolution's weight norm.

As a dependency of another pixi project (MAX still comes from Modular's conda channel):

```toml
[workspace]
channels = ["https://conda.modular.com/max/", "conda-forge"]
platforms = ["osx-arm64"]

[dependencies]
python = "3.14.*"
max = ">=26.6.0,<27"

[pypi-dependencies]
nadakit = { git = "https://github.com/alix559/nadakit.git" }
```


## Transcribe

```sh
pixi run transcribe clip.wav
```

The clip must be 16 kHz, 16-bit PCM WAV. The same thing from Python:

```python
from nadakit.features import read_wav
from nadakit.model import build, compile_model, load_checkpoint, transcribe

clip = read_wav("clip.wav")
model = build()
load_checkpoint(model)
compiled = compile_model(model, batch=1, num_samples=clip.size)
print(transcribe(model, compiled, [clip])[0])
```


## Embed

```python
from nadakit import Engine, trim
import numpy as np

engine = Engine()          # loads ~/.cache/nadakit/models/wav2vec2.safetensors
wave = np.zeros(16_000, dtype=np.float32)
vec = engine.embed(wave)   # 768-d unit vector
```


## Run the app

```sh
pixi run serve
```

The FastHTML app listens on <http://127.0.0.1:5001>. On macOS, `pixi install -e app` adds the Quartz and CoreBluetooth bindings used to type into other apps and to send keys to a PC over Bluetooth LE (`nadakit-receive`).

This is an [nbdev](https://nbdev.fast.ai) project. Library code lives in notebooks under `nbs/` and is exported to the `nadakit` package.


## Train a head

Recognition starts as nearest example. Train, on the Sound Names page, fits a linear classifier to every stored embedding and ranks with it from then on. The `0.6` cosine check still decides whether a glyph is typed.

`nbs/13_torch_embed.ipynb` is the PyTorch wav2vec that produces the same embedding as the MAX engine. `nbs/14_learn.ipynb` fits that head with fastai and writes the same `head.npz` the button writes. Both notebooks run in a separate environment, so the app process stays on MAX:

```sh
pixi install -e train
```
